# JavaScript 迁移与 JSDoc

学习目标：在保留 JavaScript 行为测试的前提下，引入 JSDoc 检查并逐步迁移为 TypeScript。

前置知识：JavaScript 模块和注释、函数默认参数、泛型、类型与值的区别。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/21-javascript-migration/。

1. [legacy.js](scripts/21-javascript-migration/legacy.js)：带 JSDoc 的既有模块。
2. [model.ts](scripts/21-javascript-migration/model.ts)：迁移后的数据接口与函数。
3. [main.ts](scripts/21-javascript-migration/main.ts)：TS 消费 JS 的混合入口。
4. [legacy.test.js](scripts/21-javascript-migration/legacy.test.js)：迁移前后复用的行为测试。
5. [tsconfig.json](scripts/21-javascript-migration/tsconfig.json)：混合项目的正常配置。
6. [type-errors.js](scripts/21-javascript-migration/type-errors.js)：独立的类型反例；配置为 tsconfig.errors.json。

## 1 让 JavaScript 进入检查项目

迁移不必一次改完扩展名。allowJs 让 JS 文件成为项目输入；checkJs 在这些文件内报告类型错误。只打开 allowJs 并不等于所有 JS 都已检查。文件级的 // @ts-check 可用于按文件启用检查，本例直接对列出的 JS 开启 checkJs。

outDir 与源码分开，防止生成 JavaScript 时覆盖原文件。files 列出入口，入口所导入的 model.ts 也在同一项目内。strict 仍显式开启，未迁移的文件不会因此自动获得完整的类型信息。

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "rootDir": ".",
    "outDir": "./.build",
    "noEmitOnError": true,
    "allowJs": true,
    "checkJs": true
  },
  "files": [
    "legacy.js",
    "model.ts",
    "main.ts",
    "legacy.test.js"
  ]
}
```

Step 1：检查本章正常项目。

```bash
npm run check:21
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:21
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:21
# 输出 笔:10 11 gross 7，并运行两项既有行为测试。
```

## 2 用注释声明数据与导入类型

@import 从另一个模块引入仅供 JSDoc 使用的名字，不会运行该模块；这里的 Product 来自 model.ts。输出仍写 .js 扩展名，让编译后的相对导入可被 Node.js 加载。真正调用 label 时需要 main.ts 中的运行时 import。

@type 为变量声明类型；@typedef 为结构命名。@satisfies 检查值满足 Policy，同时不把变量直接标成整个 Policy 类型。本例 mode 保留为具体的 "gross"，不是强制断言，也不验证运行时输入。

以下片段来自 legacy.js。

```javascript
/** @import { Product } from "./model.js" */
/** @typedef {{ mode: "gross" | "net", rate: number }} Policy */
/** @type {Product} */
export const product = { name: "笔", price: 10 };
/** @satisfies {Policy} */
export const policy = { mode: "gross", rate: 0.1 };
```

以下片段来自 model.ts。

```typescript
export interface Product { name: string; price: number }
export function label(product: Product): string {
  return product.name + ":" + product.price;
}
```

## 3 参数、返回值与回调类型

@param 的名字必须对应实际形参。方括号 [rate=0] 表示有默认值的可选参数，真正的默认行为来自函数签名中的 rate = 0，注释本身不执行赋值。@returns 描述返回结果。

@callback 为可复用的函数签名命名，再用 @type 给回调提供上下文类型，round 的 value 因此是 number。金额例子只观察注释与计算关系，不承担货币精度方案。

```javascript
/**
 * @param {number} price
 * @param {number} [rate=0]
 * @returns {number}
 */
export function total(price, rate = 0) {
  return price * (1 + rate);
}

/**
 * @callback PriceRule
 * @param {number} value
 * @returns {number}
 */
/** @type {PriceRule} */
export const round = value => Math.round(value);
```

## 4 泛型注释连接输入与输出

@template T 声明类型参数；T 代表本次调用的输入类型，identity 返回同一个 T，而不是丢失关系的宽泛类型。传入 7 后，结果可以赋给 number；传入对象时，运行时仍返回原引用。

JS 中的注释类型只供工具读取。把单个文件迁移为 .ts 后，可把相同关系改写成 TS 类型参数和标注，其余 JS 可以继续保留。

```javascript
/**
 * @template T
 * @param {T} value
 * @returns {T}
 */
export function identity(value) { return value; }
```

以下片段来自 main.ts。

```typescript
import { product, policy, total, round, identity } from "./legacy.js";
import { label } from "./model.js";
const mode: "gross" = policy.mode;
const selected: number = identity(7);
// 预期：笔:10 11 gross 7；JS 的泛型关系与字面量约束传到 TS 调用方。
console.log(label(product), round(total(product.price, policy.rate)), mode, selected);
```

## 5 核对 JavaScript 推断的版本差异

不要把历史 JavaScript 检查规则当作 TS 7 的承诺。旧手册中“函数参数通常可省略”、自动把值名当类型名等便利规则，需结合 7.0 的变更阅读。本章的 JS 与 TS 使用更一致的参数检查；unknown 参数仍是必需参数，只有可选写法或默认值等条件才允许这里省略。

JS 仍不能直接写冒号类型标注，参数、类型别名等信息依靠 JSDoc。旧 @enum、@class 构造函数注释及 Closure 函数类型写法已有变更；新写法优先使用类、@typedef 和 TS 风格函数类型，不靠旧注释改变运行语义。

legacy.js 中另有 `/** @param {unknown} value */` 标注的 `keep(value)`，直接返回 value；unknown 表示参数的值类型，不表示可省略参数。三个反例分别观察实参、参数个数和值名误作类型名。

以下片段来自 type-errors.js。

```javascript
import { total, keep } from "./legacy.js";
total("10"); // TS2345：string 不是 number。
keep(); // TS2554：TS 7 不因参数是 unknown 就允许省略。
const sample = { name: "笔" };
/** @type {sample} */ // TS2749：值查询应写 typeof sample。
const other = { name: "纸" };
export {};
```

Step 1：单独检查类型反例。

```bash
npm run errors:21
# TS 7.0.2 退出 1；包含 TS2345、TS2554、TS2749。
```

## 6 逐步收紧与保留行为测试

先把现有行为固定为可重复测试，再对模块边界补 JSDoc，修正当前检查诊断，最后逐个迁移扩展名。迁移时避免让同目录的同名 .js 与 .ts 同时成为竞争输入；移除被替代文件后核对导入和输出位置。

优先处理外部输入和公共函数，不用 @ts-ignore 或 any 批量消音。strict 和 checkJs 通过后，按需求增加索引或可选属性检查，逐项修复并重新运行原有测试。这里既检查默认税率，也检查对象引用身份，确保注释或迁移没有悄悄改变可观察行为。

以下片段来自 legacy.test.js。

```javascript
import test from "node:test";
import assert from "node:assert/strict";
import { total, identity } from "./legacy.js";
test("旧行为：默认税率和指定税率", () => {
  assert.equal(total(10), 10);
  assert.equal(total(10, 0.1), 11);
  assert.equal(total(0, 0.1), 0);
});
test("泛型注释不改变引用身份", () => {
  const item = { id: 3 };
  assert.equal(identity(item), item);
});
```

## 本章小结

- allowJs 决定输入范围，checkJs 决定 JS 内部的检查；配置中仍要隔离输出。
- JSDoc 能表达类型、函数和泛型关系，但不会创建运行时校验或导入。
- 迁移以小模块为单位，类型检查与原有行为测试一起保留。

## 练习

1. 为 total 新增第三个折扣函数参数，并用 @callback 描述；默认函数原样返回金额，折扣应用于含税结果。以 price=10、rate=0 比较默认与半价折扣，核对 10 和 5，原有测试继续通过。
2. 将 identity 单独迁移到 .ts 并更新导入；字符串、数值、对象三类调用都通过检查，对象身份测试仍通过。
3. 修正 type-errors.js 三处错误后重查：没有 TS2345、TS2554、TS2749；说明 typeof sample 与 sample 各处于什么位置。

### 提示

1. 保留 rate 作为第二参数，新增第三参数默认值。
2. 同步修改 main.ts 与 legacy.test.js 的导入位置。
3. 依次修改实参类型、补参数、使用 typeof 查询类型。

### 参考解析

1. 定义输入/输出均为 number 的回调，第三参数默认 `value => value`，返回 `discount(price * (1 + rate))`。默认行为不变；rate=0 时半价回调 `value => value / 2` 得到 5。
2. 新文件中声明 `identity&lt;T&gt;(value: T): T`（T 连接输入与输出），删除 legacy.js 中被替代的实现并更新调用方导入；既有对象引用测试无需改变断言。
3. 改为 `total(10)`、`keep(undefined)`、`@type {typeof sample}`。sample 在表达式中是运行时值；typeof sample 在该类型位置查询其结构。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [迁移 JS](https://www.typescriptlang.org/docs/handbook/migrating-from-javascript.html#setting-up-your-directories)、[allowJs](https://www.typescriptlang.org/tsconfig/allowJs.html)、[checkJs](https://www.typescriptlang.org/tsconfig/checkJs.html)；[JSDoc Reference](https://www.typescriptlang.org/docs/handbook/jsdoc-supported-types.html) 的 @type、@import、@param、@returns、@typedef、@callback、@template、@satisfies 小节：混合项目与注释语法。历史 JS 推断对照 [Type Checking JavaScript Files](https://www.typescriptlang.org/docs/handbook/type-checking-javascript-files.html)，具体差异按 7.0 说明处理。 |
| Microsoft Developer Blogs | [TypeScript 7.0 / JavaScript Differences](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/#javascript-differences)：旧注释和值类型查询的变化。 |
| GitHub / Microsoft | [typescript-go CHANGES.md / JavaScript support](https://github.com/microsoft/typescript-go/blob/main/CHANGES.md#javascript-support)：参数检查、构造函数与 JSDoc 的迁移条件。 |
| Node.js 24.11.0 | [测试运行器](https://nodejs.org/download/release/v24.11.0/docs/api/test.html#test-runner)、[严格断言](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html#strict-assertion-mode)：保留旧行为测试。 |
